# 13 — Dòng tiền nhà đầu tư

"Khối ngoại mua ròng 500 tỷ" là câu xuất hiện mỗi ngày trên báo. Notebook này
dựng lại con số đó từ dữ liệu gốc, rồi chỉ ra ba chỗ nó bị dùng sai:

1. **Bốn nhóm chi tiết cộng lại bằng 0** — cộng chúng với `foreign` là đếm hai lần
2. **`foreign` đến từ nguồn khác** với `foreign_individual + foreign_institutional`
3. **Bốn nhóm chi tiết chỉ có ở HOSE**, và chỉ từ 2024

Rồi dùng nó thật: dòng tiền theo ngành, và mối liên hệ với giá.

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import pandas as pd

import finlens
from finlens_examples import (
    ap_dung_theme,
    bar_ngang,
    duong,
    heatmap,
    hom_nay,
    lui_ngay,
    thanh_doi_mau,
    ty_dong,
)

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

## 1 · Sáu nhóm, ba nguồn, hai trục

`investor.breakdown()` trả về tất cả các nhóm cùng lúc, dạng long.

In [2]:
hpg = client.eod.stock.investor.breakdown("HPG", start=lui_ngay(HOM_NAY, thang=6))

print("Các nhóm có mặt:", sorted(hpg["group"].unique()))
print("Đơn vị:", {k: v for k, v in hpg.attrs["finlens"]["units"].items() if v})
hpg.head(3)

Các nhóm có mặt: ['foreign', 'foreign_individual', 'foreign_institutional', 'local_individual', 'local_institutional', 'proprietary']
Đơn vị: {'buy_value': 'VND', 'sell_value': 'VND', 'net_value': 'VND', 'buy_volume': 'share', 'sell_volume': 'share', 'net_volume': 'share'}


,symbol,date,group,buy_value,sell_value,net_value,buy_volume,sell_volume,net_volume
0,HPG,2026-02-12,foreign,8.860429e+10,6.309425e+10,2.551004e+10,3290910.0,2343425.0,947485.0
1,HPG,2026-02-12,foreign_individual,3.562900e+09,4.308201e+10,-3.951911e+10,132100.0,1599400.0,-1467300.0
2,HPG,2026-02-12,foreign_institutional,8.512369e+10,1.870249e+10,6.642121e+10,3158810.0,693949.0,2464861.0


| Nhóm | Có từ | Phạm vi | Nguồn |
|---|---|---|---|
| `foreign` | 2010 | cả ba sàn | A |
| `proprietary` (tự doanh) | 2022 | cả ba sàn | A |
| `foreign_individual`, `foreign_institutional` | 2024 | **chỉ HOSE** | B |
| `local_individual`, `local_institutional` | 2024 | **chỉ HOSE** | B |

Một mã HNX chỉ có hai nhóm — kiểm chứng luôn:

In [3]:
hnx = client.eod.stock.investor.breakdown("SHS", start=lui_ngay(HOM_NAY, thang=1))
print(f"SHS (HNX) có: {sorted(hnx['group'].unique())}")

SHS (HNX) có: ['foreign', 'proprietary']


## 2 · Bốn nhóm chi tiết cộng lại bằng 0

Đây không phải một đặc tính kỳ lạ, nó là định nghĩa: **mua ròng của nhóm này
chính là bán ròng của nhóm kia**. Bốn nhóm chi tiết chia hết toàn bộ giao dịch
trên HOSE, nên tổng của chúng luôn triệt tiêu.

In [4]:
bang = hpg.pivot_table(index="date", columns="group", values="net_value").dropna()

BON_NHOM = [
    "foreign_individual",
    "foreign_institutional",
    "local_individual",
    "local_institutional",
]

tong = bang[BON_NHOM].sum(axis=1)
print(f"Tổng bốn nhóm chi tiết qua {len(bang)} phiên:")
print(f"  trung vị |tổng| = {tong.abs().median():,.0f} VND")
print(f"  lớn nhất |tổng| = {tong.abs().max():,.0f} VND   ← chỉ là sai số làm tròn")

Tổng bốn nhóm chi tiết qua 114 phiên:
  trung vị |tổng| = 0 VND
  lớn nhất |tổng| = 2 VND   ← chỉ là sai số làm tròn


Nên **`foreign + foreign_individual + foreign_institutional` là một phép cộng
sai**. Nó cộng một đại lượng với chính phần cấu thành của nó, ở hai nguồn khác
nhau. Kết quả là một con số không đo cái gì cả.

Hai nhóm `foreign` và `proprietary` nằm trên **trục riêng** — chúng đến từ
nguồn A và không chia hết thị trường.

### `foreign` gần bằng nhưng không bằng tổng hai nhóm ngoại

Đây là chỗ dễ vấp nhất: hai con số sát nhau đủ để bạn tưởng chúng là một.

In [5]:
lech = (bang["foreign_individual"] + bang["foreign_institutional"]) - bang["foreign"]

print(f"(foreign_individual + foreign_institutional) − foreign, {len(bang)} phiên:")
print(f"  trung vị |lệch| = {lech.abs().median():>18,.0f} VND")
print(f"  lớn nhất |lệch| = {lech.abs().max():>18,.0f} VND")
print(f"  quy mô foreign  = {bang['foreign'].abs().median():>18,.0f} VND (trung vị)")
print(f"\n  → lệch tương đối trung vị: {(lech.abs() / bang['foreign'].abs()).median():.3%}")
print(f"  → nhưng có phiên lệch tới {ty_dong(lech.abs().max())} tỷ đồng")

(foreign_individual + foreign_institutional) − foreign, 114 phiên:
  trung vị |lệch| =        110,188,398 VND
  lớn nhất |lệch| =     68,830,357,656 VND
  quy mô foreign  =     51,220,150,272 VND (trung vị)

  → lệch tương đối trung vị: 0.192%
  → nhưng có phiên lệch tới 68.8 tỷ đồng


Lệch tương đối trung vị dưới 0,3% nghe như sai số làm tròn — cho tới khi bạn
gặp cái phiên lệch mấy chục tỷ. **Chọn một nguồn và đứng nguyên ở đó**: `foreign` nếu bạn cần
chuỗi dài từ 2010 và cả ba sàn, cặp `foreign_*` nếu bạn cần tách cá nhân với tổ
chức và chấp nhận chỉ có HOSE từ 2024.

## 3 · Vẽ đúng: bốn nhóm chi tiết là một bức tranh tổng bằng 0

In [6]:
ve = bang[BON_NHOM].div(1e9).reset_index().melt(
    id_vars="date", var_name="nhom", value_name="mua_rong_ty"
)
ten_viet = {
    "foreign_individual": "Ngoại · cá nhân",
    "foreign_institutional": "Ngoại · tổ chức",
    "local_individual": "Nội · cá nhân",
    "local_institutional": "Nội · tổ chức",
}
ve["nhom"] = ve["nhom"].map(ten_viet)

tich_luy = ve.sort_values("date").assign(
    tich_luy=lambda d: d.groupby("nhom", observed=True)["mua_rong_ty"].cumsum()
)

duong(
    tich_luy,
    x="date",
    y="tich_luy",
    theo="nhom",
    tieu_de="HPG — mua ròng tích luỹ theo bốn nhóm nhà đầu tư",
    phu_de="Bốn đường luôn triệt tiêu nhau: tiền một nhóm bỏ ra là tiền nhóm khác thu về",
    nhan_y="tỷ đồng, tích luỹ",
    moc_khong=True,
)

Bốn đường phản chiếu nhau quanh trục 0 — đó chính là tính chất tổng-bằng-0 hiện
ra bằng hình. Cái đọc được ở đây không phải "ai đang mua", mà **ai đang bán cho
ai**: tổ chức nước ngoài gom thì cá nhân trong nước là bên bán ra.

## 4 · Khối ngoại theo phiên — chuỗi dài, cả ba sàn

Khi cần lịch sử dài hoặc cần mã ngoài HOSE, dùng `investor.flow()` với
`group="foreign"`.

In [7]:
ngoai = client.eod.stock.investor.flow("HPG", group="foreign", start=lui_ngay(HOM_NAY, thang=3))
ngoai = ngoai.assign(mua_rong_ty=ty_dong(ngoai["net_value"]))

thanh_doi_mau(
    ngoai.tail(30),
    x="date",
    y="mua_rong_ty",
    tieu_de="HPG — khối ngoại mua/bán ròng, 30 phiên gần nhất",
    phu_de="Đơn vị gốc là VND; ở đây đổi sang tỷ đồng để đọc được bằng mắt",
    nhan_y="tỷ đồng",
    dinh_dang_nhan="{:+,.0f}",
)

## 5 · Dòng tiền theo ngành — 19 ngành trong một request

Đây là góc nhìn hữu dụng nhất trong thực tế: tiền ngoại đang rút khỏi ngành nào
và chảy vào ngành nào.

In [8]:
cap2 = client.meta.sectors(level=2)

dong_nganh = client.eod.sector.investor.flow(
    cap2["icb"].tolist(),
    icb_level=2,
    group="foreign",
    start=lui_ngay(HOM_NAY, thang=1),
)

xep = (
    dong_nganh.groupby("icb_name", observed=True)["net_value"]
    .sum()
    .div(1e9)
    .round(0)
    .reset_index()
    .rename(columns={"net_value": "mua_rong_ty"})
)

bar_ngang(
    xep,
    nhan="icb_name",
    gia_tri="mua_rong_ty",
    tieu_de="Khối ngoại mua/bán ròng theo ngành — một tháng",
    phu_de="Cộng dồn giá trị ròng của từng ngành ICB cấp 2",
    nhan_x="tỷ đồng",
    dinh_dang_nhan="{:+,.0f}",
)

### Theo tuần: dòng tiền có bền không hay chỉ một phiên đột biến

Một ngành +2.000 tỷ có thể là mua đều bốn tuần, hoặc một phiên thoả thuận duy
nhất. Heatmap tách hai câu chuyện đó ra.

In [9]:
theo_tuan = dong_nganh.assign(tuan=dong_nganh["date"].dt.to_period("W").astype(str))
bang_tuan = (
    theo_tuan.groupby(["icb_name", "tuan"], observed=True)["net_value"]
    .sum()
    .div(1e9)
    .unstack()
    .round(0)
)
bang_tuan.columns = [c[-5:] for c in bang_tuan.columns]  # chỉ giữ ngày cuối tuần
bang_tuan = bang_tuan.loc[bang_tuan.sum(axis=1).sort_values(ascending=False).index]

heatmap(
    bang_tuan,
    tieu_de="Khối ngoại theo ngành và theo tuần",
    phu_de="Một ô đậm đơn độc = giao dịch đột biến, không phải xu hướng",
    nhan_mau="tỷ đồng",
    dinh_dang_o="%{z:,.0f}",
)

## 6 · Dòng tiền có đi trước giá không?

Câu hỏi ai cũng hỏi. Đo thẳng: tương quan giữa mua ròng của khối ngoại phiên
`t` với lợi suất phiên `t`, `t+1`, `t+2`… trên rổ VN30.

⚠️ Đây là **tương quan, không phải nhân quả**, và một hệ số cao ở độ trễ 0 chỉ
nói rằng khối ngoại mua vào phiên giá tăng — điều đó không dùng để giao dịch
được, vì bạn chỉ biết số liệu sau khi phiên đã đóng.

In [10]:
ma_hose = client.meta.symbols(exchange="HOSE", kind="stock")
# Lấy 30 mã HOSE thanh khoản nhất làm rổ đại diện
gia_1n = client.eod.stock.ohlcv(ma_hose["symbol"].tolist(), start=lui_ngay(HOM_NAY, nam=1))
thanh_khoan = (
    gia_1n.assign(gtgd=gia_1n["close"] * gia_1n["volume"] * 1_000)
    .groupby("symbol", observed=True)["gtgd"]
    .mean()
    .nlargest(30)
)
RO = thanh_khoan.index.tolist()
print(f"Rổ 30 mã thanh khoản nhất HOSE: {', '.join(RO[:12])}…")

Rổ 30 mã thanh khoản nhất HOSE: SHB, HPG, SSI, FPT, VIX, VPB, VIC, MBB, MWG, MSN, TCB, STB…


C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


In [11]:
gia_ro = gia_1n[gia_1n["symbol"].isin(RO)].sort_values(["symbol", "date"])
gia_ro = gia_ro.assign(
    ls=gia_ro.groupby("symbol", observed=True)["close"].pct_change() * 100
)

dong_ro = client.eod.stock.investor.flow(RO, group="foreign", start=lui_ngay(HOM_NAY, nam=1))

ghep = gia_ro.merge(
    dong_ro[["symbol", "date", "net_value"]], on=["symbol", "date"], how="inner"
).dropna(subset=["ls", "net_value"])

# Chuẩn hoá mua ròng theo giá trị giao dịch của chính mã đó — 100 tỷ ở VCB
# không cùng ý nghĩa với 100 tỷ ở một mã nhỏ.
ghep = ghep.assign(gtgd=ghep["close"] * ghep["volume"] * 1_000)
ghep = ghep.assign(mua_rong_chuan=ghep["net_value"] / ghep["gtgd"])

ket = []
for do_tre in range(0, 6):
    tam = ghep.sort_values(["symbol", "date"]).copy()
    tam["ls_tuong_lai"] = tam.groupby("symbol", observed=True)["ls"].shift(-do_tre)
    r = tam[["mua_rong_chuan", "ls_tuong_lai"]].dropna().corr().iloc[0, 1]
    ket.append({"do_tre": f"t+{do_tre}", "tuong_quan": round(r, 4)})

tuong_quan = pd.DataFrame(ket)
print(tuong_quan.to_string(index=False))

thanh_doi_mau(
    tuong_quan,
    x="do_tre",
    y="tuong_quan",
    tieu_de="Tương quan: mua ròng ngoại phiên t với lợi suất phiên t+k",
    phu_de=f"Rổ 30 mã HOSE thanh khoản nhất · {len(ghep):,} quan sát · 12 tháng",
    nhan_y="hệ số tương quan",
    dinh_dang_nhan="{:+.3f}",
)

do_tre  tuong_quan
   t+0      0.1844
   t+1     -0.0019
   t+2      0.0144
   t+3     -0.0207
   t+4     -0.0162
   t+5     -0.0034


Hệ số ở `t+0` cao hơn hẳn các độ trễ sau — khối ngoại **mua trong phiên giá
tăng**, chứ tín hiệu không kéo dài sang phiên sau. Đó là kết quả thường gặp và
nó nói rằng dòng tiền là thứ để *giải thích* phiên vừa rồi, không phải để *dự
báo* phiên tới.

## 7 · Top mua/bán ròng toàn sàn — bảng dùng hằng ngày

In [12]:
toan_sa = client.eod.stock.investor.flow(
    ma_hose["symbol"].tolist(), group="foreign", start=lui_ngay(HOM_NAY, ngay=10)
)
phien_cuoi = toan_sa["date"].max()
hom_qua = toan_sa[toan_sa["date"] == phien_cuoi]

top = pd.concat(
    [hom_qua.nlargest(10, "net_value"), hom_qua.nsmallest(10, "net_value")]
).assign(mua_rong_ty=lambda d: ty_dong(d["net_value"]))

bar_ngang(
    top,
    nhan="symbol",
    gia_tri="mua_rong_ty",
    tieu_de=f"Top khối ngoại mua/bán ròng — phiên {phien_cuoi:%d/%m/%Y}",
    phu_de="Toàn bộ cổ phiếu HOSE",
    nhan_x="tỷ đồng",
    dinh_dang_nhan="{:+,.1f}",
)

C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


## Tổng kết

| Bạn cần | Gọi |
|---|---|
| Khối ngoại một mã, chuỗi dài | `eod.stock.investor.flow("HPG", group="foreign")` |
| Tất cả các nhóm cùng lúc | `eod.stock.investor.breakdown("HPG")` |
| Dòng tiền theo ngành | `eod.sector.investor.flow(icb, icb_level=2)` |
| Tự doanh công ty chứng khoán | `group="proprietary"` |

**Ba điều mang sang notebook sau:**

1. **Bốn nhóm chi tiết cộng lại bằng 0.** Cộng chúng với `foreign` là đếm hai
   lần. Chọn một nguồn và đứng nguyên ở đó.
2. `foreign` lệch với `foreign_individual + foreign_institutional` chưa tới
   0,3% ở trung vị — nhưng có phiên lệch hàng chục tỷ. Hai nguồn khác nhau.
3. Mua ròng tuyệt đối không so được giữa các mã. **Chia cho giá trị giao dịch**
   của chính mã đó trước khi xếp hạng hay tính tương quan.

---

**Tiếp theo:** [`14_dashboard_hang_ngay.ipynb`](14_dashboard_hang_ngay.ipynb) —
ghép Track 1 thành một báo cáo thị trường một trang, xuất ra HTML.